<a href="https://colab.research.google.com/github/herr-hollis/F1-Bahrain-2026-Power-BI/blob/main/F1_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastf1 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 424.7/424.7 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 6.4 MB/s eta 0:00:00


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import requests
import fastf1

OUT, CACHE = "data", "f1_cache"
JOLPICA = "https://api.jolpi.ca/ergast/f1"
OPENF1 = "https://api.openf1.org/v1"
os.makedirs(OUT, exist_ok=True)
os.makedirs(CACHE, exist_ok=True)
fastf1.Cache.enable_cache(CACHE)
fastf1.set_log_level("WARNING")


def save(df, name):
    df.to_csv(os.path.join(OUT, f"{name}.csv"), index=False)
    print(f"  saved {name}.csv  ({len(df):,} rows)")


def td_to_s(df):
    for c in df.columns:
        if pd.api.types.is_timedelta64_dtype(df[c]):
            df[c] = df[c].dt.total_seconds()
    return df


def get_json(url, params=None, tries=4):
    for i in range(tries):
        r = requests.get(url, params=params, timeout=60)
        if r.status_code == 429:          # rate-limited: back off and retry
            time.sleep(2 * (i + 1))
            continue
        r.raise_for_status()
        return r.json()
    r.raise_for_status()


def of1(endpoint, **params):
    """OpenF1 request -> DataFrame. Keys like 'date>=' become raw range filters
    (OpenF1 expects e.g. date>=2026-10-04T07:00:00, unencoded). 404 = no rows."""
    parts = []
    for k, v in params.items():
        parts.append(f"{k}{v}" if k.endswith((">=", "<=")) else f"{k}={v}")
    url = f"{OPENF1}/{endpoint}" + ("?" + "&".join(parts) if parts else "")
    time.sleep(0.4)
    for i in range(4):
        r = requests.get(url, timeout=60)
        if r.status_code == 429:
            time.sleep(2 * (i + 1))
            continue
        if r.status_code == 404:
            return pd.DataFrame()
        r.raise_for_status()
        return pd.DataFrame(r.json())
    r.raise_for_status()


def of1_time(ts):
    """OpenF1-friendly UTC timestamp (no timezone suffix)."""
    return pd.Timestamp(ts).tz_convert("UTC").strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]


def jolpica(path, key):
    pages, offset, total = [], 0, 1
    while offset < total:
        mr = get_json(f"{JOLPICA}/{path}", {"limit": 100, "offset": offset})["MRData"]
        total = int(mr["total"])
        pages.append(mr.get("RaceTable", mr.get("StandingsTable", {})).get(key, []))
        offset += 100
        time.sleep(0.3)
    return pages


def add_minisectors(tel):
    tel["MiniSector"] = (tel["Distance"] / tel["Distance"].max() * 25).clip(upper=24.999).astype(int) + 1
    return tel


# ======================================================================= FastF1 path
def fastf1_session(rnd, code):
    s = fastf1.get_session(2026, rnd, code)
    s.load(laps=True, telemetry=True, weather=True, messages=True)
    try:
        laps = s.laps
        if laps is None or laps.empty:
            raise ValueError
    except Exception:
        return s, None
    return s, laps


def export_fastf1(s, laps, label, is_race):
    res = s.results[["DriverNumber", "Abbreviation", "FullName", "TeamName", "TeamColor",
                     "GridPosition", "Position", "Status", "Points", "Q1", "Q2", "Q3", "Time"]].copy()
    res["TeamColor"] = "#" + res["TeamColor"].astype(str)
    if is_race:
        res["PositionsGained"] = res["GridPosition"] - res["Position"]
    save(td_to_s(res), f"{label}_results")

    L = laps.copy()
    L["IsClean"] = (L["PitInTime"].isna() & L["PitOutTime"].isna()
                    & (L["TrackStatus"].astype(str) == "1") & L["IsAccurate"].fillna(False))
    keep = ["Driver", "DriverNumber", "Team", "LapNumber", "LapTime", "Sector1Time", "Sector2Time",
            "Sector3Time", "SpeedI1", "SpeedI2", "SpeedST", "Compound", "TyreLife", "Stint",
            "Position", "TrackStatus", "IsClean"]
    save(td_to_s(L[keep].copy()), f"{label}_laps")

    tel_all = []
    for drv in L["Driver"].dropna().unique():
        try:
            lap = laps.pick_drivers(drv).pick_fastest()
            t = lap.get_telemetry()[["Distance", "Speed", "Throttle", "Brake", "nGear", "X", "Y"]]
            t = t.assign(Distance=(t["Distance"] // 10) * 10).groupby("Distance", as_index=False).mean()
            t["Driver"] = drv
            tel_all.append(t)
        except Exception as e:
            print(f"    telemetry skipped for {drv}: {e}")
    if tel_all:
        save(add_minisectors(pd.concat(tel_all, ignore_index=True)), f"{label}_telemetry")

    if is_race:
        st = (laps.dropna(subset=["Stint"]).groupby(["Driver", "Stint", "Compound"], as_index=False)
              .agg(StartLap=("LapNumber", "min"), EndLap=("LapNumber", "max"), Laps=("LapNumber", "count")))
        save(st, "race_stints")
        save(td_to_s(s.weather_data.copy()), "race_weather")
        save(s.race_control_messages.copy(), "race_control")


# ======================================================================= OpenF1 path
def find_openf1_session(name):
    ss = of1("sessions", year=2026)
    if not ss.empty:
        ss["date_start"] = pd.to_datetime(ss["date_start"], utc=True, format="ISO8601")
        ss = ss[(ss["date_start"] >= "2026-10-01") & (ss["date_start"] <= "2026-10-06")
                & (ss["session_name"].str.lower() == name.lower())]
    if ss.empty:
        raise RuntimeError(f"OpenF1 has no '{name}' session for 2–6 Oct 2026 yet.")
    return int(ss.iloc[0]["session_key"])


def export_openf1(name, label, is_race, results_ff1=None):
    sk = find_openf1_session(name)
    print(f"  OpenF1 session_key = {sk}")

    drv = of1("drivers", session_key=sk)[["driver_number", "name_acronym", "full_name", "team_name", "team_colour"]]
    drv = drv.drop_duplicates("driver_number")
    drv["team_colour"] = "#" + drv["team_colour"].astype(str)
    acr = dict(zip(drv["driver_number"], drv["name_acronym"]))
    team = dict(zip(drv["driver_number"], drv["team_name"]))

    if results_ff1 is not None:          # results already came from FastF1/Jolpica
        res = results_ff1
    else:
        res = drv.rename(columns={"driver_number": "DriverNumber", "name_acronym": "Abbreviation",
                                  "full_name": "FullName", "team_name": "TeamName", "team_colour": "TeamColor"})
    save(res, f"{label}_results")

    laps = of1("laps", session_key=sk)
    laps["date_start"] = pd.to_datetime(laps["date_start"], utc=True, format="ISO8601")
    pits = of1("pit", session_key=sk)
    stints = of1("stints", session_key=sk)

    # tyre compound + tyre age per lap from stints
    laps["Compound"], laps["TyreLife"], laps["Stint"] = None, np.nan, np.nan
    for _, st in stints.iterrows():
        m = (laps["driver_number"] == st["driver_number"]) & laps["lap_number"].between(st["lap_start"], st["lap_end"])
        laps.loc[m, "Compound"] = st["compound"]
        laps.loc[m, "Stint"] = st["stint_number"]
        laps.loc[m, "TyreLife"] = st["tyre_age_at_start"] + (laps.loc[m, "lap_number"] - st["lap_start"]) + 1

    # position at the end of each lap (race only)
    laps["Position"] = np.nan
    if is_race:
        pos = of1("position", session_key=sk)
        pos["date"] = pd.to_datetime(pos["date"], utc=True, format="ISO8601")
        laps["lap_end"] = laps["date_start"] + pd.to_timedelta(laps["lap_duration"].fillna(0), unit="s")
        has_t = laps["lap_end"].notna()
        merged = pd.merge_asof(laps[has_t].sort_values("lap_end"),
                               pos.sort_values("date")[["date", "driver_number", "position"]],
                               left_on="lap_end", right_on="date", by="driver_number", direction="backward")
        merged = merged.drop(columns=["date", "Position"]).rename(columns={"position": "Position"})
        laps = pd.concat([merged, laps[~has_t]], ignore_index=True)

    # clean-lap flag: no pit in/out, within 107% of the driver's median lap
    pit_in = set(zip(pits["driver_number"], pits["lap_number"])) if not pits.empty else set()
    med = laps.groupby("driver_number")["lap_duration"].transform("median")
    laps["IsClean"] = (~laps["is_pit_out_lap"].fillna(False).astype(bool)
                       & ~laps.apply(lambda r: (r["driver_number"], r["lap_number"]) in pit_in, axis=1)
                       & (laps["lap_duration"] <= med * 1.07)
                       & ((laps["lap_number"] > 1) | (not is_race)))

    out = pd.DataFrame({
        "Driver": laps["driver_number"].map(acr), "DriverNumber": laps["driver_number"],
        "Team": laps["driver_number"].map(team), "LapNumber": laps["lap_number"],
        "LapTime": laps["lap_duration"], "Sector1Time": laps["duration_sector_1"],
        "Sector2Time": laps["duration_sector_2"], "Sector3Time": laps["duration_sector_3"],
        "SpeedI1": laps["i1_speed"], "SpeedI2": laps["i2_speed"], "SpeedST": laps["st_speed"],
        "Compound": laps["Compound"], "TyreLife": laps["TyreLife"], "Stint": laps["Stint"],
        "Position": laps["Position"], "IsClean": laps["IsClean"]})
    save(out.sort_values(["Driver", "LapNumber"]), f"{label}_laps")

    # telemetry: fastest lap per driver (car_data + location), resampled every 10 m
    tel_all = []
    for dn, g in laps.dropna(subset=["lap_duration"]).groupby("driver_number"):
        f = g.loc[g["lap_duration"].idxmin()]
        t0, t1 = f["date_start"], f["date_start"] + pd.to_timedelta(f["lap_duration"], unit="s")
        rng = {"date>=": of1_time(t0), "date<=": of1_time(t1)}
        try:
            car = of1("car_data", session_key=sk, driver_number=int(dn), **rng)
            loc = of1("location", session_key=sk, driver_number=int(dn), **rng)
            if car.empty:
                continue
            for d in (car, loc):
                d["date"] = pd.to_datetime(d["date"], utc=True, format="ISO8601")
            car = car.sort_values("date")
            dt = car["date"].diff().dt.total_seconds().fillna(0)
            car["Distance"] = (car["speed"] / 3.6 * dt).cumsum()
            if not loc.empty:
                car = pd.merge_asof(car, loc.sort_values("date")[["date", "x", "y"]], on="date", direction="nearest")
            else:
                car["x"], car["y"] = np.nan, np.nan
            t = car.rename(columns={"speed": "Speed", "throttle": "Throttle", "brake": "Brake",
                                    "n_gear": "nGear", "x": "X", "y": "Y"})
            t = t[["Distance", "Speed", "Throttle", "Brake", "nGear", "X", "Y"]]
            t = t.assign(Distance=(t["Distance"] // 10) * 10).groupby("Distance", as_index=False).mean()
            t["Driver"] = acr.get(dn, dn)
            tel_all.append(t)
        except Exception as e:
            print(f"    telemetry skipped for #{dn}: {e}")
    if tel_all:
        save(add_minisectors(pd.concat(tel_all, ignore_index=True)), f"{label}_telemetry")

    if is_race:
        st = stints.rename(columns={"driver_number": "DriverNumber", "stint_number": "Stint",
                                    "compound": "Compound", "lap_start": "StartLap", "lap_end": "EndLap",
                                    "tyre_age_at_start": "TyreAgeStart"})
        st["Driver"] = st["DriverNumber"].map(acr)
        st["Laps"] = st["EndLap"] - st["StartLap"] + 1
        save(st[["Driver", "DriverNumber", "Stint", "Compound", "StartLap", "EndLap", "Laps", "TyreAgeStart"]], "race_stints")
        if not pits.empty:
            pits["Driver"] = pits["driver_number"].map(acr)
            save(pits, "race_pitstops")
        save(of1("weather", session_key=sk), "race_weather")
        save(of1("race_control", session_key=sk), "race_control")


# ======================================================================= main
schedule = fastf1.get_event_schedule(2026)
event = schedule[schedule["EventName"].str.contains("Bahrain", case=False)].iloc[0]
rnd = int(event["RoundNumber"])
print(f"2026 event: {event['EventName']} | Round {rnd} | {event['Location']}")

for code, of_name, label in [("R", "Race", "race"), ("Q", "Qualifying", "quali")]:
    print(f"\n=== 2026 {label} ===")
    s, laps = fastf1_session(rnd, code)
    if laps is not None:
        print("  using FastF1")
        export_fastf1(s, laps, label, code == "R")
    else:
        print("  FastF1 has no lap data -> falling back to OpenF1")
        res = None
        try:
            res = s.results[["DriverNumber", "Abbreviation", "FullName", "TeamName", "TeamColor",
                             "GridPosition", "Position", "Status", "Points", "Q1", "Q2", "Q3"]].copy()
            res["TeamColor"] = "#" + res["TeamColor"].astype(str)
            if code == "R":
                res["PositionsGained"] = res["GridPosition"] - res["Position"]
            res = td_to_s(res)
        except Exception:
            pass
        try:
            export_openf1(of_name, label, code == "R", res)
        except Exception as e:
            print(f"  OpenF1 failed: {e}")
            if res is not None:
                save(res, f"{label}_results")
            print(f"  -> {label} lap/telemetry data not published yet; continuing with the rest.")

print("\n=== 2026 standings ===")
d = jolpica(f"2026/{rnd}/driverStandings.json", "StandingsLists")[0][0]["DriverStandings"]
save(pd.DataFrame([{"Position": int(x["position"]), "Driver": x["Driver"].get("code"),
                    "Points": float(x["points"]), "Wins": int(x["wins"]),
                    "Team": x["Constructors"][0]["name"]} for x in d]), "standings_drivers")
c = jolpica(f"2026/{rnd}/constructorStandings.json", "StandingsLists")[0][0]["ConstructorStandings"]
save(pd.DataFrame([{"Position": int(x["position"]), "Team": x["Constructor"]["name"],
                    "Points": float(x["points"]), "Wins": int(x["wins"])} for x in c]), "standings_teams")

print("\n=== 2017 Malaysian GP (Sepang) ===")
rnd17 = int(get_json(f"{JOLPICA}/2017/circuits/sepang/races.json")["MRData"]["RaceTable"]["Races"][0]["round"])


def to_s(t):
    if not t:
        return None
    m, s_ = (t.split(":") if ":" in t else ("0", t))
    return int(m) * 60 + float(s_)


q = jolpica(f"2017/{rnd17}/qualifying.json", "Races")[0][0]["QualifyingResults"]
save(pd.DataFrame([{"Position": int(x["position"]), "Driver": x["Driver"].get("code"),
                    "Team": x["Constructor"]["name"], "Q1": to_s(x.get("Q1")),
                    "Q2": to_s(x.get("Q2")), "Q3": to_s(x.get("Q3"))} for x in q]), "sepang2017_quali")
rows = []
for page in jolpica(f"2017/{rnd17}/laps.json", "Races"):
    for race in page:
        for lap in race["Laps"]:
            for t in lap["Timings"]:
                rows.append({"LapNumber": int(lap["number"]), "DriverId": t["driverId"],
                             "Position": int(t["position"]), "LapTime": to_s(t["time"])})
save(pd.DataFrame(rows), "sepang2017_laps")

print("\nDone. CSVs are in ./data")

INFO:fastf1.fastf1.core:Loading data for Bahrain Grand Prix - Race [v3.8.3]
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
INFO:fastf1.api:Fetching session info data...
DEBUG:fastf1.api:Falling back to livetiming mirror (https://livetiming-mirror.fastf1.dev)


2026 event: Bahrain Grand Prix | Round 16 | Kuala Lumpur

=== 2026 race ===


logger      WARNING 	Failed to load session info data!
DEBUG:fastf1.fastf1.core:Traceback for failure in session info data
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/fastf1/logger.py", line 112, in __wrapped
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/fastf1/core.py", line 1452, in _load_session_info
    self._session_info = api.session_info(self.api_path,
                         ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
                                          livedata=livedata)
                                          ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/fastf1/req.py", line 459, in _cached_api_request
    data = func(api_path, **func_kwargs)
  File "/usr/local/lib/python3.13/dist-packages/fastf1/_api.py", line 1730, in session_info
    raise SessionNotAvailableError(
    ...<2 lines>...
    )
fastf1._api.SessionNotAvailableError: No data for this session! If this session only finished

  FastF1 has no lap data -> falling back to OpenF1
  OpenF1 session_key = 11731
  saved race_results.csv  (22 rows)
  saved race_laps.csv  (1,145 rows)
  saved race_telemetry.csv  (7,432 rows)
  saved race_stints.csv  (97 rows)
  saved race_pitstops.csv  (73 rows)
  saved race_weather.csv  (254 rows)


INFO:fastf1.fastf1.core:Loading data for Bahrain Grand Prix - Qualifying [v3.8.3]
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
INFO:fastf1.api:Fetching session info data...


  saved race_control.csv  (328 rows)

=== 2026 quali ===


DEBUG:fastf1.api:Falling back to livetiming mirror (https://livetiming-mirror.fastf1.dev)
logger      WARNING 	Failed to load session info data!
DEBUG:fastf1.fastf1.core:Traceback for failure in session info data
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/fastf1/logger.py", line 112, in __wrapped
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/fastf1/core.py", line 1452, in _load_session_info
    self._session_info = api.session_info(self.api_path,
                         ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
                                          livedata=livedata)
                                          ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/fastf1/req.py", line 459, in _cached_api_request
    data = func(api_path, **func_kwargs)
  File "/usr/local/lib/python3.13/dist-packages/fastf1/_api.py", line 1730, in session_info
    raise SessionNotAvailableError(
    ...<2 lines>...
    )
fas

  FastF1 has no lap data -> falling back to OpenF1
  OpenF1 session_key = 11730
  saved quali_results.csv  (22 rows)
  saved quali_laps.csv  (239 rows)
    telemetry skipped for #3: 429 Client Error: Too Many Requests for url: https://api.openf1.org/v1/location?session_key=11730&driver_number=3&date%3E=2026-10-03T08:58:39.243&date%3C=2026-10-03T09:00:14.373
    telemetry skipped for #55: 429 Client Error: Too Many Requests for url: https://api.openf1.org/v1/car_data?session_key=11730&driver_number=55&date%3E=2026-10-03T08:39:24.087&date%3C=2026-10-03T08:41:01.614
  saved quali_telemetry.csv  (6,630 rows)

=== 2026 standings ===
  saved standings_drivers.csv  (23 rows)
  saved standings_teams.csv  (11 rows)

=== 2017 Malaysian GP (Sepang) ===
  saved sepang2017_quali.csv  (20 rows)
  saved sepang2017_laps.csv  (1,024 rows)

Done. CSVs are in ./data


In [ ]:
!zip -r f1_data.zip data
from google.colab import files
files.download("f1_data.zip")

  adding: data/ (stored 0%)
  adding: data/race_laps.csv (deflated 73%)
  adding: data/race_telemetry.csv (deflated 76%)
  adding: data/race_results.csv (deflated 52%)
  adding: data/race_weather.csv (deflated 81%)
  adding: data/race_control.csv (deflated 88%)
  adding: data/quali_results.csv (deflated 46%)
  adding: data/race_pitstops.csv (deflated 74%)
  adding: data/race_stints.csv (deflated 72%)
  adding: data/sepang2017_laps.csv (deflated 75%)
  adding: data/standings_drivers.csv (deflated 50%)
  adding: data/quali_telemetry.csv (deflated 76%)
  adding: data/standings_teams.csv (deflated 32%)
  adding: data/quali_laps.csv (deflated 69%)
  adding: data/sepang2017_quali.csv (deflated 49%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import re
import time
import tempfile
from collections import Counter

import subprocess
import sys

import pandas as pd
import requests

try:
    import whisper
except ModuleNotFoundError:          # install on first run (package is "openai-whisper")
    print("Installing openai-whisper (one-time, ~1 min)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai-whisper"])
    import whisper

OUT = "data"
OPENF1 = "https://api.openf1.org/v1"
SESSION = "Race"            # or "Qualifying"
MODEL = "small"             # "medium" = more accurate, slower; "base" = fastest
os.makedirs(OUT, exist_ok=True)


def of1(endpoint, **params):
    q = "&".join(f"{k}={v}" for k, v in params.items())
    time.sleep(0.4)
    r = requests.get(f"{OPENF1}/{endpoint}?{q}", timeout=60)
    if r.status_code == 404:
        return pd.DataFrame()
    r.raise_for_status()
    return pd.DataFrame(r.json())


# ---------------------------------------------------------------- find the session
ss = of1("sessions", year=2026)
ss["date_start"] = pd.to_datetime(ss["date_start"], utc=True, format="ISO8601")
ss = ss[(ss["date_start"] >= "2026-10-01") & (ss["date_start"] <= "2026-10-06")
        & (ss["session_name"] == SESSION)]
if ss.empty:
    raise SystemExit(f"OpenF1 has no {SESSION} session for Sepang yet — try again later.")
sk = int(ss.iloc[0]["session_key"])

drivers = of1("drivers", session_key=sk).drop_duplicates("driver_number")
acr = dict(zip(drivers["driver_number"], drivers["name_acronym"]))

radio = of1("team_radio", session_key=sk)
if radio.empty:
    raise SystemExit("No team radio clips published for this session yet.")
radio["date"] = pd.to_datetime(radio["date"], utc=True, format="ISO8601")
print(f"{len(radio)} radio clips across {radio['driver_number'].nunique()} drivers")

# lap number for each clip (latest lap started before the clip)
laps = of1("laps", session_key=sk)
if not laps.empty:
    laps["date_start"] = pd.to_datetime(laps["date_start"], utc=True, format="ISO8601")
    radio = pd.merge_asof(radio.sort_values("date"),
                          laps.dropna(subset=["date_start"]).sort_values("date_start")[["date_start", "driver_number", "lap_number"]],
                          left_on="date", right_on="date_start", by="driver_number", direction="backward")
else:
    radio["lap_number"] = None

# ---------------------------------------------------------------- transcribe
model = whisper.load_model(MODEL)
F1_PROMPT = ("Formula 1 team radio. Box box, push, copy, mode push, overtake, DRS, deg, "
             "softs, mediums, hards, inters, safety car, VSC, gap, delta, undercut, "
             "lift and coast, battery, harvest, deploy, P1, Sepang, turn 1, turn 15.")

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/126.0 Safari/537.36",
           "Referer": "https://www.formula1.com/"}
shown_diag = False

rows = []
for i, r in radio.reset_index(drop=True).iterrows():
    text = ""
    try:
        resp = requests.get(r["recording_url"], headers=HEADERS, timeout=60, allow_redirects=True)
        audio = resp.content
        # a real clip is tens of KB of MP3 (starts with ID3 or an MPEG frame), not an HTML/XML error page
        if resp.status_code != 200 or len(audio) < 5000 or audio[:1] in (b"<", b"{"):
            if not shown_diag:
                print(f"  ! clip URL did not return audio: HTTP {resp.status_code}, "
                      f"{resp.headers.get('Content-Type')}, {len(audio)} bytes\n"
                      f"    URL: {r['recording_url']}\n    start: {audio[:150]!r}")
                shown_diag = True
            raise ValueError("not an audio file")
        with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as f:
            f.write(audio)
            path = f.name
        text = model.transcribe(path, language="en", initial_prompt=F1_PROMPT, fp16=False)["text"].strip()
        os.remove(path)
    except Exception as e:
        print(f"  clip {i} failed: {str(e).strip().splitlines()[-1][:120]}")
    rows.append({"Driver": acr.get(r["driver_number"], r["driver_number"]),
                 "DriverNumber": r["driver_number"], "Time": r["date"], "Lap": r["lap_number"],
                 "Transcript": text, "AudioURL": r["recording_url"]})
    print(f"  [{i + 1}/{len(radio)}] {rows[-1]['Driver']}: {text[:70]}")

clips = pd.DataFrame(rows)
clips.to_csv(os.path.join(OUT, "race_team_radio.csv"), index=False)
if (clips["Transcript"] == "").all():
    raise SystemExit("\nNo clip could be transcribed — paste the '! clip URL did not return audio' lines above to Claude.")

# ---------------------------------------------------------------- word counts per driver
STOP = set("""
a an the and or but if so to of in on at for with from by as is are was were be been being it its
this that these those i you he she we they me him her us them my your our their mine yours
do does did done have has had will would can could should shall may might must just
yeah yes no ok okay right um uh ah oh hmm mate guys guy got get go going gonna wanna
there here what when where which who how all any some very really now then than too
about up down out over again still well also not dont don't im i'm it's that's you're we're
let lets let's one two three thing things bit like know see say said think
""".split())

wc = []
for drv, g in clips.groupby("Driver"):
    words = re.findall(r"[a-zA-Z][a-zA-Z0-9']+", " ".join(g["Transcript"]).lower())
    for w, n in Counter(w for w in words if w not in STOP and len(w) > 2).items():
        wc.append({"Driver": drv, "Word": w, "Count": n})
pd.DataFrame(wc).sort_values(["Driver", "Count"], ascending=[True, False]) \
    .to_csv(os.path.join(OUT, "race_radio_words.csv"), index=False)

print("\nSaved race_team_radio.csv and race_radio_words.csv in ./data")

40 radio clips across 11 drivers
  ! clip URL did not return audio: HTTP 403, text/html, 919 bytes
    URL: https://livetiming.formula1.com/static/2026/2026-10-04_Bahrain_Grand_Prix/2026-10-04_Race/TeamRadio/RUS_63_20261004_142745.mp3
    start: b'<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">\n<HTML><HEAD><META HTTP-EQUIV="Content-Type" CON'
  clip 0 failed: not an audio file
  [1/40] RUS: 
  clip 1 failed: not an audio file
  [2/40] VER: 
  clip 2 failed: not an audio file
  [3/40] NOR: 
  clip 3 failed: not an audio file
  [4/40] ANT: 
  clip 4 failed: not an audio file
  [5/40] ANT: 
  clip 5 failed: not an audio file
  [6/40] HAM: 
  clip 6 failed: not an audio file
  [7/40] LIN: 
  clip 7 failed: not an audio file
  [8/40] LEC: 
  clip 8 failed: not an audio file
  [9/40] LEC: 
  clip 9 failed: not an audio file
  [10/40] VER: 
  clip 10 failed: not an audio file
  [11/40] HAM: 
  clip 11 failed: not an audio file
  [12/40] 

SystemExit: 
No clip could be transcribed — paste the '! clip URL did not return audio' lines above to Claude.

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
